# openapi_pydantic smoke test

Exercises `openapi_pydantic` 0.5.x on its current pydantic-v2 API: programmatic spec
construction, dict validation with `$ref` handling, Pydantic-model schema generation via
`openapi_pydantic.util`, alias-preserving round-trips, and `oneOf` + `Discriminator`
polymorphism.

In [ ]:
# Example 1: Build an OpenAPI spec object programmatically (current pydantic-v2 API).
from openapi_pydantic import Info, MediaType, OpenAPI, Operation, PathItem, Response, Schema

spec = OpenAPI(
    info=Info(title="Ping API", version="1.0.0"),
    paths={
        "/ping": PathItem(
            get=Operation(
                operationId="ping",
                responses={
                    "200": Response(
                        description="pong",
                        content={"application/json": MediaType(media_type_schema=Schema(type="object"))},
                    )
                },
            )
        )
    },
)

print(f"openapi version: {spec.openapi}")
print(f"title: {spec.info.title} v{spec.info.version}")
assert spec.openapi.startswith("3.1")
assert spec.paths["/ping"].get.operationId == "ping"

In [ ]:
# Example 2: Validate a raw dict spec — $ref containers parse into Reference objects.
from openapi_pydantic import OpenAPI, Reference, Schema

raw = {
    "openapi": "3.1.1",
    "info": {"title": "User API", "version": "2.0.0"},
    "paths": {},
    "components": {
        "schemas": {
            "User": {
                "type": "object",
                "required": ["id"],
                "properties": {
                    "id": {"type": "integer"},
                    "manager": {"$ref": "#/components/schemas/User"},
                },
            }
        }
    },
}

spec = OpenAPI.model_validate(raw)
user = spec.components.schemas["User"]
print(f"User properties: {list(user.properties)}")
assert isinstance(user, Schema)
assert isinstance(user.properties["manager"], Reference)
assert user.properties["manager"].ref == "#/components/schemas/User"

In [ ]:
# Example 3: Generate component schemas from a Pydantic model via openapi_pydantic.util.
from pydantic import BaseModel

from openapi_pydantic import Info, MediaType, OpenAPI, Operation, PathItem, RequestBody, Response
from openapi_pydantic.util import PydanticSchema, construct_open_api_with_schema_class

class PingRequest(BaseModel):
    name: str
    retries: int = 3

base = OpenAPI(
    info=Info(title="Util API", version="0.1.0"),
    paths={
        "/ping": PathItem(
            post=Operation(
                requestBody=RequestBody(
                    content={"application/json": MediaType(media_type_schema=PydanticSchema(schema_class=PingRequest))}
                ),
                responses={"200": Response(description="ok")},
            )
        )
    },
)

spec = construct_open_api_with_schema_class(base)
assert "PingRequest" in spec.components.schemas
props = spec.components.schemas["PingRequest"].properties
print(f"generated schemas: {list(spec.components.schemas)}")
print(f"PingRequest properties: {list(props)}")
assert set(props) == {"name", "retries"}

In [ ]:
# Example 4: Round-trip — dump with aliases ($ref) then re-validate to an equal model.
# In OpenAPI 3.1 a $ref under components.schemas is carried by the Schema model (JSON
# Schema 2020-12 semantics) as an extra field, so assert at the dump level.
import json

from openapi_pydantic import Info, OpenAPI, Operation, PathItem, Response

spec = OpenAPI(
    info=Info(title="RT API", version="1.2.3"),
    paths={"/x": PathItem(get=Operation(responses={"200": Response(description="ok")}))},
    components={"schemas": {"Alias": {"$ref": "#/components/schemas/Real"}}},
)

dumped = spec.model_dump(mode="json", by_alias=True, exclude_none=True)
assert dumped["components"]["schemas"]["Alias"] == {"$ref": "#/components/schemas/Real"}

text = json.dumps(dumped)
reparsed = OpenAPI.model_validate_json(text)
assert reparsed == spec
assert reparsed.model_dump(mode="json", by_alias=True, exclude_none=True) == dumped
print(f"round-trip OK — {len(text)} bytes; $ref alias preserved")

In [ ]:
# Example 5: Polymorphism — oneOf + Discriminator.
from openapi_pydantic import DataType, Discriminator, Reference, Schema

pet = Schema(
    oneOf=[Reference(ref="#/components/schemas/Cat"), Reference(ref="#/components/schemas/Dog")],
    discriminator=Discriminator(
        propertyName="petType",
        mapping={"cat": "#/components/schemas/Cat", "dog": "#/components/schemas/Dog"},
    ),
)

assert pet.discriminator.propertyName == "petType"
dumped = pet.model_dump(mode="json", by_alias=True, exclude_none=True)
print(dumped)
assert [r["$ref"] for r in dumped["oneOf"]] == ["#/components/schemas/Cat", "#/components/schemas/Dog"]
assert dumped["discriminator"]["propertyName"] == "petType"

# DataType enumerates the JSON Schema primitive types.
print(f"DataType members: {[d.value for d in DataType]}")
assert Schema(type=DataType.STRING).type == DataType.STRING